In [1]:
import time
import struct
import numpy as np
import scipy.ndimage
from pynq import Overlay, allocate

ol = Overlay("conv2d_designfloat_wrapper.bit")
print("Overlay loaded")
print("IP blocks:", list(ol.ip_dict.keys()))

dma         = ol.axi_dma_0
conv_stream  = ol.conv2d_0
# fir_mmio_ip = ol.fir_mmio_0
hw_timer    = ol.axi_timer_0

Overlay loaded
IP blocks: ['conv2d_0', 'axi_timer_0', 'axi_dma_0', 'processing_system7_0']


In [2]:
inp_size = 64
kernel_size = 3
N_SAMPLES = inp_size * inp_size

img = np.random.randint(low=0, high=255, size=(inp_size, inp_size))/255
kernel = np.random.uniform(-1, 1, (kernel_size, kernel_size))
# kernel = [[0, 1, 0], [-1, 1, 1], [2, -2, 3]]
t0=time.perf_counter()
exp_out = scipy.ndimage.convolve(img, kernel, mode='constant', cval=0.0)
t_now = time.perf_counter() - t0
print(t_now)

0.002822834000085095


In [3]:
def float_to_raw_int(f_val):
    return struct.unpack('<I', struct.pack('<f', float(f_val)))[0]

In [4]:
in_buf  = allocate(shape=(N_SAMPLES,), dtype=np.float32)
out_buf = allocate(shape=(N_SAMPLES,), dtype=np.float32)

np.copyto(in_buf, list(img.flatten()))
out_buf[:] = 0
# 1. Write the weights using the helper function!
conv_stream.write(0x10, float_to_raw_int(kernel[0][0]))
conv_stream.write(0x18, float_to_raw_int(kernel[0][1]))
conv_stream.write(0x20, float_to_raw_int(kernel[0][2]))

conv_stream.write(0x28, float_to_raw_int(kernel[1][0]))
conv_stream.write(0x30, float_to_raw_int(kernel[1][1]))
conv_stream.write(0x38, float_to_raw_int(kernel[1][2]))

conv_stream.write(0x40, float_to_raw_int(kernel[2][0]))
conv_stream.write(0x48, float_to_raw_int(kernel[2][1]))
conv_stream.write(0x50, float_to_raw_int(kernel[2][2]))

In [5]:
# 2. Write the 1D image size
# conv_stream.write(0x58, inp_size)

conv_stream.write(0x58, inp_size)

# Configure streaming FIR: set sample count and start
# Register 0x10 = 'n' parameter (check synthesis report)
# conv_stream.write(0x10, N_SAMPLES)
conv_stream.write(0x00, 0x01)   # ap_start

t0 = time.perf_counter()
dma.sendchannel.transfer(in_buf)
dma.recvchannel.transfer(out_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()
t_dma = time.perf_counter() - t0

y_dma = np.array(out_buf, dtype=np.float32)
print(f"DMA: {N_SAMPLES} samples in {t_dma*1e3:.2f} ms "
      f"({t_dma/N_SAMPLES*1e6:.1f} us/sample)")

DMA: 4096 samples in 3.17 ms (0.8 us/sample)


In [6]:
print("y_dma:", y_dma)

y_dma: [-0.07374638 -0.77575684 -0.765693   ... -0.29915142 -0.20374595
  0.36754704]


In [7]:
np.max(y_dma - exp_out.flatten()) * 255

5.0242926279420175e-05

In [8]:
y_dma.shape

(4096,)

In [9]:
128 * 128

16384

In [10]:
TCSR0, TLR0, TCR0 = 0x00, 0x04, 0x08
FCLK_MHZ = 100.0

def timer_start(tmr):
    tmr.write(TLR0, 0)
    tmr.write(TCSR0, 0x020)   # load
    tmr.write(TCSR0, 0x080)   # enable, count up

def timer_stop(tmr):
    cycles = tmr.read(TCR0)
    tmr.write(TCSR0, 0x000)
    return cycles

In [11]:
conv_stream.write(0x00, 0x01)   # ap_start
dma.sendchannel.start()
dma.recvchannel.start()
# Trigger DMA transfer
timer_start(hw_timer)
dma.sendchannel.transfer(in_buf)
dma.recvchannel.transfer(out_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()
cycles = timer_stop(hw_timer)

y_dma = np.array(out_buf, dtype=np.float32)

total_us = cycles / FCLK_MHZ
per_sample_us = total_us / N_SAMPLES

print(f"HW timer: {cycles} cycles = {total_us:.3f} us total @ {FCLK_MHZ:.0f} MHz")
print(f"HW: {N_SAMPLES} samples in {total_us/1e3:.2f} ms ({per_sample_us:.3f} us/sample)")

HW timer: 542708 cycles = 5427.080 us total @ 100 MHz
HW: 4096 samples in 5.43 ms (1.325 us/sample)


In [12]:
np.max(y_dma - exp_out.flatten()) * 255

5.0242926279420175e-05

In [40]:
# import matplotlib.pyplot as plt

# if(if_random == 0):
#     # Reshape outputs back to 2D
#     img_display = img  # already (128, 128), float
#     hw_out_2d   = y_dma.reshape(inp_size, inp_size)
#     sw_out_2d   = exp_out  # scipy reference output

#     # Difference map
#     diff = np.abs(hw_out_2d - sw_out_2d)

#     fig, axes = plt.subplots(1, 4, figsize=(18, 4))

#     axes[0].imshow(img_display, cmap='gray')
#     axes[0].set_title('Input Image')
#     axes[0].axis('off')

#     axes[1].imshow(hw_out_2d, cmap='gray')
#     axes[1].set_title('HW Output (FPGA)')
#     axes[1].axis('off')

#     axes[2].imshow(sw_out_2d, cmap='gray')
#     axes[2].set_title('SW Output (SciPy)')
#     axes[2].axis('off')

#     im = axes[3].imshow(diff, cmap='hot_r')
#     axes[3].set_title(f'|HW - SW| (max={diff.max():.4f})')
#     axes[3].axis('off')
#     plt.colorbar(im, ax=axes[3])

#     plt.tight_layout()
#     plt.savefig("conv_comparison.png", dpi=150, bbox_inches='tight')
#     plt.show()

#     print(f"Max error : {diff.max():.6f}")
#     print(f"Mean error: {diff.mean():.6f}")
#     print(f"MSE       : {(diff**2).mean():.8f}")